# AB 2GPU Architecture Search

This notebook tests new landmark-aware model architectures. The main candidate is `HybridLandmarkMixer`, which treats the 478 MediaPipe face landmarks as tokens instead of flattening everything into one MLP input.

In [ ]:
import json
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(r"c:\\Users\\ldy34\\Desktop\\Face")
ML = BASE / "ML"
VENV_PY = BASE / ".venv" / "Scripts" / "python.exe"
TRAIN_PY = ML / "train_landmark_arch_2gpu_ddp.py"
OUT_ROOT = ML / "experiments_arch_2gpu"
OUT_ROOT.mkdir(exist_ok=True)

print("trainer exists:", TRAIN_PY.exists())
print("out root:", OUT_ROOT)


In [ ]:
BASE_CFG = {
    "base_dir": str(BASE / "video"),
    "cache_path": str(ML / "cache_landmarks_7class.npz"),
    "force_rebuild_cache": False,
    "num_gpus": 2,
    "seed": 123,
    "batch_size_per_gpu": 256,
    "loader_workers": 4,
    "extract_workers": 12,
    "max_per_zip": 5000,
    "test_size": 0.15,
    "val_size": 0.15,
    "epochs_binary_direct": 140,
    "epochs_pretrain7": 180,
    "epochs_binary_finetune": 120,
    "lr_binary_direct": 2e-4,
    "lr_pretrain7": 6e-4,
    "lr_binary_finetune": 2e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.01,
    "noise_std": 0.0005,
    "early_stop_patience": 35,
    "early_stop_min_delta": 1e-4,
    "amp": True,
}

TARGET_TEST_ACC = 0.852
BASE_CFG


In [ ]:
ARCH_EXPERIMENTS = [
    {
        "name": "hybrid_mixer_d128_l6",
        "overrides": {
            "arch": "hybrid_mixer",
            "d_model": 128,
            "mixer_layers": 6,
            "token_mlp_dim": 128,
            "channel_mlp_dim": 256,
            "model_dropout": 0.15,
            "global_branch_dim": 128,
        },
    },
    {
        "name": "hybrid_mixer_d160_l5",
        "overrides": {
            "arch": "hybrid_mixer",
            "d_model": 160,
            "mixer_layers": 5,
            "token_mlp_dim": 160,
            "channel_mlp_dim": 320,
            "model_dropout": 0.15,
            "global_branch_dim": 128,
            "batch_size_per_gpu": 192,
        },
    },
    {
        "name": "token_mixer_d160_l6",
        "overrides": {
            "arch": "token_mixer",
            "d_model": 160,
            "mixer_layers": 6,
            "token_mlp_dim": 160,
            "channel_mlp_dim": 320,
            "model_dropout": 0.15,
            "batch_size_per_gpu": 192,
        },
    },
    {
        "name": "transformer_lite_d96_l2",
        "overrides": {
            "arch": "transformer_lite",
            "d_model": 96,
            "transformer_layers": 2,
            "num_heads": 4,
            "channel_mlp_dim": 256,
            "model_dropout": 0.15,
            "batch_size_per_gpu": 96,
        },
    },
]

pd.DataFrame([{"name": e["name"], **e["overrides"]} for e in ARCH_EXPERIMENTS])


In [ ]:
def build_cmd(cfg, out_dir):
    cmd = [
        str(VENV_PY), str(TRAIN_PY),
        "--base-dir", cfg["base_dir"],
        "--out-dir", str(out_dir),
        "--cache-path", cfg["cache_path"],
        "--arch", cfg["arch"],
        "--num-gpus", str(cfg["num_gpus"]),
        "--seed", str(cfg["seed"]),
        "--batch-size-per-gpu", str(cfg["batch_size_per_gpu"]),
        "--loader-workers", str(cfg["loader_workers"]),
        "--extract-workers", str(cfg["extract_workers"]),
        "--max-per-zip", str(cfg["max_per_zip"]),
        "--test-size", str(cfg["test_size"]),
        "--val-size", str(cfg["val_size"]),
        "--epochs-binary-direct", str(cfg["epochs_binary_direct"]),
        "--epochs-pretrain7", str(cfg["epochs_pretrain7"]),
        "--epochs-binary-finetune", str(cfg["epochs_binary_finetune"]),
        "--lr-binary-direct", str(cfg["lr_binary_direct"]),
        "--lr-pretrain7", str(cfg["lr_pretrain7"]),
        "--lr-binary-finetune", str(cfg["lr_binary_finetune"]),
        "--weight-decay", str(cfg["weight_decay"]),
        "--d-model", str(cfg["d_model"]),
        "--mixer-layers", str(cfg.get("mixer_layers", 6)),
        "--token-mlp-dim", str(cfg.get("token_mlp_dim", 128)),
        "--channel-mlp-dim", str(cfg["channel_mlp_dim"]),
        "--num-heads", str(cfg.get("num_heads", 4)),
        "--transformer-layers", str(cfg.get("transformer_layers", 2)),
        "--model-dropout", str(cfg["model_dropout"]),
        "--global-branch-dim", str(cfg.get("global_branch_dim", 128)),
        "--label-smoothing", str(cfg["label_smoothing"]),
        "--noise-std", str(cfg["noise_std"]),
        "--early-stop-patience", str(cfg["early_stop_patience"]),
        "--early-stop-min-delta", str(cfg["early_stop_min_delta"]),
    ]
    if cfg["force_rebuild_cache"]:
        cmd.append("--force-rebuild-cache")
    cmd.append("--amp" if cfg["amp"] else "--no-amp")
    return cmd


def run_experiment(exp):
    cfg = dict(BASE_CFG)
    cfg.update(exp["overrides"])
    out_dir = OUT_ROOT / exp["name"] / f"seed_{cfg['seed']}"
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / "ab_summary.json").exists():
        print("SKIP completed:", out_dir)
        return out_dir

    workers = [cfg["loader_workers"], 2, 0]
    workers = list(dict.fromkeys(workers))
    for lw in workers:
        cfg["loader_workers"] = lw
        cmd = build_cmd(cfg, out_dir)
        print("\n" + "=" * 90)
        print("RUN", exp["name"], "loader_workers=", lw)
        print("CMD:", " ".join(cmd))
        try:
            subprocess.run(cmd, cwd=str(BASE), check=True)
            return out_dir
        except subprocess.CalledProcessError:
            print("failed; trying fallback worker count")
    raise RuntimeError(f"all worker fallbacks failed: {exp['name']}")


def read_summary(exp_name, out_dir):
    with open(out_dir / "ab_summary.json", "r", encoding="utf-8") as f:
        s = json.load(f)
    rows = []
    for model_name, m in s["results"].items():
        rows.append({
            "exp": exp_name,
            "model": model_name,
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "balanced_accuracy": m["balanced_accuracy"],
            "ckpt": s["artifacts"]["direct_ckpt"] if model_name == "direct_binary" else s["artifacts"]["finetune_ckpt"],
            "out_dir": str(out_dir),
        })
    return rows


In [ ]:
rows = []
for exp in ARCH_EXPERIMENTS:
    out_dir = run_experiment(exp)
    rows.extend(read_summary(exp["name"], out_dir))

df = pd.DataFrame(rows).sort_values(["accuracy", "macro_f1"], ascending=False)
display(df)
df.to_csv(OUT_ROOT / "architecture_results.csv", index=False, encoding="utf-8-sig")
print("saved:", OUT_ROOT / "architecture_results.csv")


In [ ]:
best = df.iloc[0]
print("Best architecture model")
print(best[["exp", "model", "accuracy", "macro_f1", "balanced_accuracy", "ckpt"]])
print("target", TARGET_TEST_ACC, "exceeded:", float(best["accuracy"]) >= TARGET_TEST_ACC)


In [ ]:
best_out = Path(best["out_dir"])

def load_hist(name):
    with open(best_out / name, "r", encoding="utf-8") as f:
        return json.load(f)

h_direct = load_hist("direct_binary_best_history.json")
h_pre7 = load_hist("pretrain7_best_history.json")
h_ft = load_hist("finetune_binary_best_history.json")

plt.figure(figsize=(16, 5))
plt.subplot(1, 3, 1)
plt.plot(h_direct["val_acc"], label="direct_binary")
plt.plot(h_ft["val_acc"], label="finetune_from_7")
plt.title("Binary Val Accuracy")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(h_direct["val_macro_f1"], label="direct_binary")
plt.plot(h_ft["val_macro_f1"], label="finetune_from_7")
plt.title("Binary Val Macro-F1")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(h_pre7["val_macro_f1"], label="pretrain7")
plt.title("7-Class Pretrain Val Macro-F1")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()
